# PySpark Benchmark

This notebook runs an representative end to end use case over data sourced from the [UK Land Registry House Price Data open data repository](https://www.gov.uk/government/statistical-data-sets/price-paid-data-downloads).

This data is made available for us under an [Open Government Licence](https://www.nationalarchives.gov.uk/doc/open-government-licence/version/3/).

We will run two processes:

1. Load raw data, clean it up, add new features and finally write it as a mini dimensional model to lakehouse.
1. Query two of the tables in the dimensional model, join them and summarise the data.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType
import time
import logging

In [2]:
logger = logging.getLogger(name="pyspark_benchmark_notebook")
logger.setLevel(logging.INFO)

In [3]:
# Initialize Spark session (on Fabric this is typically pre-configured)
# For local testing, create a session with Delta Lake support
spark = (
    SparkSession.builder
    .appName("PySpark Benchmark")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .getOrCreate()
)

JAVA_HOME is not set


PySparkRuntimeError: [JAVA_GATEWAY_EXITED] Java gateway process exited before sending its port number.

In [ ]:
from datetime import datetime

source_path = "../../data/fabric/Files/land_registry_data" # ABFSS path to location where raw data (multiple CSV files) is stored

# Add timestamp to paths to avoid overwrite issues with Parquet
run_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
target_path_prices = f"../../data/fabric/Tables/pyspark_benchmark/{run_timestamp}/prices.parquet"
target_path_locations = f"../../data/fabric/Tables/pyspark_benchmark/{run_timestamp}/locations.parquet"
target_path_dates = f"../../data/fabric/Tables/pyspark_benchmark/{run_timestamp}/dates.parquet"

In [ ]:
start = time.perf_counter()

In [ ]:
logging.info(f"Reading price paid data from location {source_path}...")

# Define schema for CSV files
schema = StructType([
    StructField("transaction_unique_identifier", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("date_of_transfer", TimestampType(), True),
    StructField("postcode", StringType(), True),
    StructField("property_type", StringType(), True),
    StructField("old_new", StringType(), True),
    StructField("duration", StringType(), True),
    StructField("paon", StringType(), True),
    StructField("saon", StringType(), True),
    StructField("street", StringType(), True),
    StructField("locality", StringType(), True),
    StructField("town_city", StringType(), True),
    StructField("district", StringType(), True),
    StructField("county", StringType(), True),
    StructField("ppd_category_type", StringType(), True),
    StructField("record_status", StringType(), True),
])

# Read CSV files - Spark can read multiple files from a directory natively
price_paid_data = (
    spark.read
    .option("header", "false")
    .option("nullValue", "")
    .schema(schema)
    .csv(source_path)
)

## Data Transformation

Now we have the DataFrame loaded, we can start to build up the transformations we want to apply using PySpark's DataFrame API:

In [ ]:
# Convert the property_type column from single letter codes to full descriptions
price_paid_data = (
    price_paid_data
    .withColumn(
        "property_type",
        F.when(F.col("property_type") == "D", F.lit("Detached"))
        .when(F.col("property_type") == "S", F.lit("Semi-Detached"))
        .when(F.col("property_type") == "T", F.lit("Terraced"))
        .when(F.col("property_type") == "F", F.lit("Flat/Maisonette"))
        .when(F.col("property_type") == "O", F.lit("Other"))
        .otherwise(F.col("property_type"))
    )
)

In [ ]:
# Do the same for old_new
price_paid_data = (
    price_paid_data
    .withColumn(
        "old_new",
        F.when(F.col("old_new") == "Y", F.lit("New"))
        .when(F.col("old_new") == "N", F.lit("Old"))
        .otherwise(F.col("old_new"))
    )
)

In [ ]:
# Use regex to extract the postcode area (the first one or two letters)
price_paid_data = (
    price_paid_data
    .withColumn(
        "postcode_area",
        F.regexp_extract(F.col("postcode"), r"^([A-Z]{1,2})", 1)
    )
)

In [ ]:
# Convert date_of_transfer from timestamp to date
price_paid_data = (
    price_paid_data
    .withColumn(
        "date_of_transfer",
        F.to_date(F.col("date_of_transfer"))
    )
)

### Create fact table

Select the core columns we want to use in the core fact table.

In [ ]:
# Select relevant columns for downstream analysis
prices = price_paid_data.select(
    "price",
    "date_of_transfer",
    "postcode_area",
    "town_city",
    "property_type",
    "old_new",
)

### Create date dimension

Use min and max dates to build date dimension table.

Spark uses lazy evaluation, so the computation is deferred until an action is triggered.

In [ ]:
date_stats = price_paid_data.agg(
    F.min("date_of_transfer").alias("min_date"),
    F.max("date_of_transfer").alias("max_date")
).collect()[0]

min_date = date_stats["min_date"]
max_date = date_stats["max_date"]
min_date, max_date

In [ ]:
# Generate date range using sequence function
dates = (
    spark.sql(f"""
        SELECT explode(sequence(
            to_date('{min_date}'),
            to_date('{max_date}'),
            interval 1 day
        )) as date
    """)
    .withColumn("year", F.year("date"))
    .withColumn("month", F.month("date"))
    .withColumn("month_name", F.date_format("date", "MMMM"))
    .withColumn("day", F.dayofmonth("date"))
    .withColumn("weekday", F.dayofweek("date") - 1)  # Adjust to 0-based (Monday=0)
    .withColumn("weekday_name", F.date_format("date", "EEEE"))
    .withColumn("day_of_year", F.dayofyear("date"))
)

### Create location dimension

Assumption is there is a hierarchy in descreasing order of granularity:

- County
- District
- Town or City

In [ ]:
locations = (
    price_paid_data
    .select(
        "county",
        "district",
        "town_city",
    )
    .distinct()
)

## Writing to Delta Tables

It is common practice to write out a PySpark DataFrame to a Delta table in the Tables area of your Lakehouse.

There are various write modes which are available:

Overwrite entire table:

```python
df.write.format("delta").mode("overwrite").save(path)
```

Append to existing table:

```python
df.write.format("delta").mode("append").save(path)
```

Merge (upsert) - use DeltaTable API:

```python
from delta.tables import DeltaTable

delta_table = DeltaTable.forPath(spark, path)
(
    delta_table.alias("target")
    .merge(
        df.alias("source"),
        "source.id = target.id"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)
```

### Handling Timestamps

A common gotcha when writing Delta tables is timezone handling. Fabric's SQL endpoint expects timestamps with timezone information.

We can address this by converting to UTC timestamp, for example:

```python
df = df.withColumn(
    "datetime_of_order",
    F.to_utc_timestamp(F.col("datetime_of_order"), "UTC")
)
```

### Write tables

In [ ]:
import os
os.makedirs(os.path.dirname(target_path_prices), exist_ok=True)

logger.info(f"Writing prices data to Parquet: {target_path_prices}")prices.write.mode("overwrite").parquet(target_path_prices)

In [ ]:
logger.info(f"Writing locations data to Parquet: {target_path_locations}")
locations.write.mode("overwrite").parquet(target_path_locations)

In [ ]:
logger.info(f"Writing dates data to Parquet: {target_path_dates}")
dates.write.mode("overwrite").parquet(target_path_dates)

## Reading from DeltaLake and generate summary

Spark uses lazy evaluation by default, so transformations are not executed until an action is triggered.

Let's illustrate this by generating some analytics in this notebook using the data we have just written to the lakehouse in Delta format.

In [ ]:
# Load prices from Parquet and filter them to exclude "Other" property types
logger.info(f"Reading prices data back from Parquet: {target_path_prices}")
prices = (
    spark.read
    .parquet(target_path_prices)
    .filter(F.col("property_type") != "Other")
)

In [ ]:
# Load the date dimension, add a new month_tag column in the form YYYY_MM
logger.info(f"Reading dates data back from Parquet: {target_path_dates}")
dates = (
    spark.read
    .parquet(target_path_dates)
    .withColumn(
        "month_tag",
        F.date_format("date", "yyyy_MM")
    )
)

In [ ]:
# Now join the two tables to get month_tag into the prices table
prices = (
    prices
    .join(
        dates.select("date", "month_tag"),
        prices["date_of_transfer"] == dates["date"],
        how="left"
    )
    .drop(dates["date"])
)

In [ ]:
# Finally summarise the data up to monthly level by property type
monthly_summary = (
    prices
    .groupBy("month_tag", "property_type")
    .agg(
        F.count("*").alias("number_of_transactions"),
        F.percentile_approx("price", 0.5).alias("median_price"),
        F.min("price").alias("min_price"),
        F.max("price").alias("max_price"),
    )
    .orderBy("month_tag", "property_type")
)

In [ ]:
# Trigger computation and convert to Pandas for display
monthly_summary_pd = monthly_summary.toPandas()

In [ ]:
monthly_summary_pd.head(5)

In [ ]:
elapsed = time.perf_counter() - start
logger.info(f"Notebook completed in {elapsed:.2f} seconds.")

## Summary

PySpark on Microsoft Fabric provides enterprise-grade distributed data processing capabilities. It's the go-to choice when working with truly large datasets that require distributed compute across multiple nodes.

While Spark has more overhead than single-node solutions like Polars or Pandas, it shines when:

- Data volumes exceed single-node memory capacity
- You need to leverage distributed compute clusters
- Integration with the broader Spark ecosystem is required
- Your organization has existing Spark expertise and infrastructure